### House Price Prediction (Linear/Identity Output)

In [21]:
# CELL 1 — Imports
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [22]:
# CELL 2 — Load California Housing dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print(df.shape)
df.head()

(20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [23]:
# CELL 3 — Feature/target split
feature_cols = housing.feature_names  # MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude
X = df[feature_cols].values.astype(np.float32)
y = df["MedHouseVal"].values.astype(np.float32) * 100000  # scale to dollars

print("Features:", feature_cols)
print("Target sample (USD):", y[:5])

Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Target sample (USD): [452600. 358500. 352100. 341300. 342200.]


In [24]:
# CELL 4 — Train/test split, scaling, tensors
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

x_scaler = StandardScaler()
X_train = x_scaler.fit_transform(X_train)
X_test = x_scaler.transform(X_test)

# Scale target too — helps training stability with Linear output
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1))
y_test_scaled = y_scaler.transform(y_test.reshape(-1, 1))

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32).to(device)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_t = torch.tensor(y_test_scaled, dtype=torch.float32).to(device)

print(X_train_t.shape, y_train_t.shape)

torch.Size([16512, 8]) torch.Size([16512, 1])


In [25]:
# CELL 5 — Model definition
class HousePriceRegressor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)  # Linear/Identity — no activation, unbounded output
        )

    def forward(self, x):
        return self.net(x)

model = HousePriceRegressor(input_dim=X_train_t.shape[1]).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(model)

HousePriceRegressor(
  (net): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [26]:
# CELL 6 — Training loop
epochs = 200
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    preds = model(X_train_t)
    loss = criterion(preds, y_train_t)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs} — Loss (scaled): {loss.item():.4f}")

Epoch 20/200 — Loss (scaled): 0.7456
Epoch 40/200 — Loss (scaled): 0.5049
Epoch 60/200 — Loss (scaled): 0.4135
Epoch 80/200 — Loss (scaled): 0.3690
Epoch 100/200 — Loss (scaled): 0.3411
Epoch 120/200 — Loss (scaled): 0.3217
Epoch 140/200 — Loss (scaled): 0.3084
Epoch 160/200 — Loss (scaled): 0.2978
Epoch 180/200 — Loss (scaled): 0.2884
Epoch 200/200 — Loss (scaled): 0.2797


In [27]:
# CELL 7 — Evaluation (unscale predictions back to dollars)
model.eval()
with torch.no_grad():
    test_preds_scaled = model(X_test_t)
    test_preds_dollars = y_scaler.inverse_transform(test_preds_scaled.cpu().numpy())
    test_actual_dollars = y_scaler.inverse_transform(y_test_t.cpu().numpy())

    mae = np.mean(np.abs(test_preds_dollars - test_actual_dollars))
    print(f"Test MAE: ${mae:,.2f}")

for i in range(5):
    print(f"Predicted: ${test_preds_dollars[i][0]:,.0f} | Actual: ${test_actual_dollars[i][0]:,.0f}")

Test MAE: $44,038.02
Predicted: $52,297 | Actual: $47,700
Predicted: $168,983 | Actual: $45,800
Predicted: $352,721 | Actual: $500,001
Predicted: $273,170 | Actual: $218,600
Predicted: $281,297 | Actual: $278,000
